# EDA – metadata her ze Steamu

## Popis dat

#### Načtení dat

In [125]:
import pandas as pd

from prepare_data import REPO, REVISION, download_metadata, load_metadata

path = download_metadata()
df = load_metadata(path)
download_date = pd.Timestamp(path.stat().st_mtime, unit="s").normalize()

print("Dataset:", REPO)
print("Revize:", REVISION)
print("Staženo:", download_date.date())

Dataset: FronkonGames/steam-games-dataset
Revize: 6c21457b9ebf0a6a7f5b91e11281d65d456ea30e
Staženo: 2026-11-15


#### Počet řádků a sloupců

In [126]:
df.shape

(148909, 42)

#### Náhled dat

In [127]:
df.head()

,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,header_image,...,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags
app_id,,,,,,,,,,,,,,,,,,,,,
2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0,0.00,0,,,,,https://shared.akamai.steamstatic.com/store_it...,...,0,0,0 - 0,0,0,0,0,0,0,[]
496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0,5.24,0,"Springtime, April: when the cherry trees come ...","Springtime, April: when the cherry trees come ...","Spring has come, and our protagonist, Yukinari...",,https://shared.akamai.steamstatic.com/store_it...,...,252,3,0 - 20000,8,0,8,0,65,0,"{'Adventure': 27, 'Visual Novel': 19, 'Anime':..."
1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0,4.99,0,"Immerse yourself in the most beloved, mystical...","Immerse yourself in the most beloved, mystical...",Discover an entrancing and spectacular world!,,https://shared.akamai.steamstatic.com/store_it...,...,21,3,0 - 20000,0,0,0,0,0,0,"{'Casual': 83, 'Card Game': 52, 'Solitaire': 4..."
3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0,8.99,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...","synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",Yuha! I'll start the broadcast! Hakko's extrem...,,https://shared.akamai.steamstatic.com/store_it...,...,0,0,0 - 20000,0,0,0,0,0,1,[]
3631080,Maze Quest VR,"Apr 24, 2025",0,4.99,0,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,,https://shared.akamai.steamstatic.com/store_it...,...,0,0,0 - 20000,0,0,0,0,0,0,[]


#### Sloupce a datové typy

In [128]:
df.info()

<class 'pandas.DataFrame'>
Index: 148909 entries, 2539430 to 2953170
Data columns (total 42 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   name                      148909 non-null  str    
 1   release_date              148909 non-null  str    
 2   required_age              148909 non-null  int64  
 3   price                     148909 non-null  float64
 4   dlc_count                 148909 non-null  int64  
 5   detailed_description      148909 non-null  str    
 6   about_the_game            148909 non-null  str    
 7   short_description         148909 non-null  str    
 8   reviews                   148909 non-null  str    
 9   header_image              148909 non-null  str    
 10  website                   148909 non-null  str    
 11  support_url               148909 non-null  str    
 12  support_email             148909 non-null  str    
 13  windows                   148909 non-null  bool   
 1

#### Pomocné sloupce
Počet recenzí a datum vydání se používají ve více buňkách, proto se počítají jednou.

In [129]:
n_reviews = df["positive"] + df["negative"]
release = pd.to_datetime(df["release_date"], format="mixed", errors="coerce")

## 1. Přehled recenzí

#### Celkový počet recenzí

In [130]:
pos = df["positive"].sum()
neg = df["negative"].sum()
total = pos + neg
print("Celkem recenzí:", total)
print(f"Pozitivní: {pos} ({pos / total:.1%})")
print(f"Negativní: {neg} ({neg / total:.1%})")

Celkem recenzí: 148914706
Pozitivní: 128157955 (86.1%)
Negativní: 20756751 (13.9%)


#### Hry s nulovým počtem recenzí
Hry s `recommendations > 0` recenze určitě mají, nula u nich tedy neznamená „žádné recenze“, ale chybějící data.

In [131]:
zero = n_reviews == 0
print("Hry s 0 recenzemi:", zero.sum())
print("  z toho recommendations > 0 (chybějící label):", (zero & (df["recommendations"] > 0)).sum())

Hry s 0 recenzemi: 65916
  z toho recommendations > 0 (chybějící label): 3467


## 2. Extrémní hodnoty

#### Numerické sloupce

In [132]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
required_age,148909.0,0.145868,1.543098,0.0,0.00,0.00,0.00,21.00
price,148909.0,5.045483,12.262702,0.0,0.74,2.99,5.99,999.99
dlc_count,148909.0,0.494960,13.232717,0.0,0.00,0.00,0.00,3703.00
metacritic_score,148909.0,2.127373,12.481296,0.0,0.00,0.00,0.00,97.00
achievements,148909.0,17.466607,128.702564,0.0,0.00,3.00,19.00,9821.00
recommendations,148909.0,798.712005,19858.223888,0.0,0.00,0.00,0.00,4830455.00
user_score,148909.0,0.020214,1.265782,0.0,0.00,0.00,0.00,100.00
positive,148909.0,860.646133,25493.862305,0.0,0.00,1.00,23.00,7642084.00
negative,148909.0,139.392186,4877.447662,0.0,0.00,0.00,6.00,1173003.00
average_playtime_forever,148909.0,171.554553,10179.359844,0.0,0.00,0.00,0.00,3429544.00


#### Nejdražší hry

In [133]:
cols = ["name", "price", "positive", "negative"]
display(df.nlargest(10, "price")[cols])

,name,price,positive,negative
app_id,,,,
4693030,Congratulations On Your Purchase,999.99,0,0
2504210,The Leverage Game Business Edition,999.98,0,0
2499620,The Leverage Game,999.98,1,0
1200520,Ascent Free-Roaming VR Experience,999.00,6,1
3013840,True Love,500.00,3,1
1105450,Aartform Curvy 3D 4.0,299.99,10,2
253670,Aartform Curvy 3D 3.0,299.90,32,13
502570,Houdini Indie,269.99,233,18
2023640,Hidden Industries Top-Down 3D,199.99,10,2


#### Zdarma hry

In [134]:
free = df["price"] == 0
print(f"Hry s cenou 0: {free.sum()} ({free.mean():.1%})")

Hry s cenou 0: 29275 (19.7%)


#### Hry s nejvíce recenzemi

In [135]:
top = df.loc[n_reviews.nlargest(10).index, cols].copy()
for c in ["positive", "negative"]:
    top[c] = [f"{v} ({v / n:.1%})" for v, n in zip(top[c], n_reviews[top.index])]
display(top)

,name,price,positive,negative
app_id,,,,
730,Counter-Strike 2,0.00,7642084 (86.7%),1173003 (13.3%)
578080,PUBG: BATTLEGROUNDS,0.00,1520457 (59.4%),1037487 (40.6%)
570,Dota 2,0.00,2037143 (81.5%),461826 (18.5%)
271590,Grand Theft Auto V Legacy,0.00,1739980 (87.4%),250576 (12.6%)
105600,Terraria,4.99,1373979 (97.5%),35494 (2.5%)
359550,Tom Clancy's Rainbow Six® Siege X,0.00,1172854 (83.9%),225730 (16.1%)
252490,Rust,19.99,1071135 (87.2%),156649 (12.8%)
440,Team Fortress 2,0.00,1044264 (89.9%),117208 (10.1%)
4000,Garry's Mod,4.99,1122546 (96.8%),37161 (3.2%)


#### Věková hranice

In [136]:
print("Hodnoty required_age:", sorted(df["required_age"].unique().tolist()))

Hodnoty required_age: [0, 1, 3, 6, 7, 8, 10, 12, 13, 14, 15, 16, 17, 18, 20, 21]


## 3. Žánry, kategorie, jazyky a screenshoty

In [137]:
counts = pd.DataFrame({
    "genres": df["genres"].map(len),
    "categories": df["categories"].map(len),
    "supported_languages": df["supported_languages"].map(len),
    "screenshots": df["screenshots"].map(len),
})
display(counts.describe().T)
print("Hry s 0 položkami:")
print((counts == 0).sum())

,count,mean,std,min,25%,50%,75%,max
genres,148909.0,2.719661,1.458011,0.0,2.0,3.0,4.0,19.0
categories,148909.0,4.623092,3.456697,0.0,2.0,4.0,6.0,32.0
supported_languages,148909.0,5.527685,13.659698,0.0,1.0,1.0,5.0,103.0
screenshots,148909.0,8.328530,5.480058,0.0,5.0,7.0,10.0,183.0


Hry s 0 položkami:
genres                 8437
categories             8981
supported_languages    8402
screenshots            6039
dtype: int64


#### Žánry

In [138]:
df["genres"].explode().value_counts()

genres
Indie                    99608
Casual                   63109
Action                   54828
Adventure                53367
Simulation               31243
Strategy                 28688
RPG                      25479
Free To Play             15181
Early Access             14951
Sports                    5768
Racing                    4726
Massively Multiplayer     3026
Utilities                 1124
Design & Illustration      596
Animation & Modeling       494
Education                  468
Violent                    414
Video Production           346
Game Development           329
Gore                       257
Audio Production           233
Software Training          225
Photo Editing              187
Web Publishing             114
Nudity                      94
Sexual Content              88
Accounting                  32
Movie                        2
Documentary                  1
Episodic                     1
Short                        1
Tutorial                     1
3

#### Software
Záznamy, jejichž žánry jsou výhradně softwarové (nejde o hry).

In [139]:
from prepare_data import is_software

print("Záznamy jen se softwarovými žánry:", df["genres"].map(is_software).sum())

ImportError: cannot import name 'is_software' from 'prepare_data' (e:\Uni\XDEEP - Projekt\Projekt\prepare_data.py)

#### Kategorie

In [ ]:
df["categories"].explode().value_counts()

categories
Single-player                        133031
Family Sharing                       120774
Steam Achievements                    70208
Steam Cloud                           40023
Full controller support               31281
                                      ...  
#category_playable_without_vision        51
Chat Speech-to-text                      43
SteamVR Collectibles                     38
Mods (require HL2)                        2
Mods                                      2
Name: count, Length: 65, dtype: int64

#### Jazyky
Nejvzácnější hodnoty ukážou případné chybné názvy jazyků.

In [ ]:
langs = df["supported_languages"].explode().value_counts()
print("Počet unikátních hodnot:", len(langs))
display(langs.tail(20))

Počet unikátních hodnot: 132


supported_languages
#lang_slovakian                                                         2
German;                                                                 2
German[b][/b]                                                           2
Italian \r\n\r\n[b][/b]                                                 1
Japanese (all with full audio support)                                  1
English\r\nRussian\r\nSpanish - Spain\r\nJapanese\r\nCzech              1
 Korean                                                                 1
\r\nFrench                                                              1
\r\nGerman                                                              1
Russian\r\nEnglish\r\nSpanish - Spain\r\nFrench\r\nJapanese\r\nCzech    1
Traditional Chinese (text only)                                         1
Hungarian,Polish                                                        1
Russian\r\n[b][/b]                                                      1
English Dutch  Eng

#### Screenshoty a header

In [ ]:
print("Hry s méně než 3 screenshoty:", (counts["screenshots"] < 3).sum())
print("Hry bez header_image:", (df["header_image"] == "").sum())

Hry s méně než 3 screenshoty: 7628
Hry bez header_image: 86


## 4. Datum vydání

#### Kontrola data vydání

In [ ]:
print("Nepřevoditelná data:", release.isna().sum())
print("Rozsah data vydání:", release.min(), "-", release.max())
print("Vydáno po datu stažení:", (release > download_date).sum())

Nepřevoditelná data: 0
Rozsah data vydání: 1997-06-30 00:00:00 - 2026-12-01 00:00:00
Vydáno po datu stažení: 1


## 5. Duplicity
Jednotkou rozdělení dat je hra – stejná hra se nesmí objevit zároveň v trénovací a testovací množině.

#### Duplicitní app_id

In [ ]:
print("Duplicitní app_id:", df.index.duplicated().sum())

Duplicitní app_id: 0


#### Stejná hra pod více app_id
Hry se stejným názvem a stejnými vývojáři pod různými `app_id`.

In [ ]:
key = df["name"].str.lower() + " | " + df["developers"].map(", ".join)
dup = df[key.duplicated(keep=False)].assign(key=key, n_reviews=n_reviews)
print(f"{dup.shape[0]} her sdílí název a vývojáře s jinou hrou:")
display(dup.sort_values("key")[["name", "release_date", "price", "n_reviews"]].head(20))

253 her sdílí název a vývojáře s jinou hrou:


,name,release_date,price,n_reviews
app_id,,,,
34019,Alpha Protocol™,"Jun 1, 2010",9.99,4045
34010,Alpha Protocol™,"Jun 1, 2010",9.99,4911
2383950,Anvil Playtest,"Apr 7, 2023",0.00,0
1648440,ANVIL Playtest,"Jun 9, 2021",0.00,0
2447870,Artifice Playtest,"Jun 12, 2023",0.00,0
2206090,Artifice Playtest,"Nov 17, 2022",0.00,0
2268670,Ascendant Playtest,"Jan 13, 2023",0.00,0
2338060,Ascendant Playtest,"Nov 14, 2024",0.00,0
48173,Assassin's Creed 2,"Mar 9, 2010",4.99,18460


## 6. Nulové recenze podle roku vydání

#### Podíl her s 0 recenzemi po letech
Sloupce tabulky pro každý rok vydání:
- `games` – počet her,
- `zero_share` – podíl her s 0 recenzemi,
- `missing_share` – podíl her s 0 recenzemi mezi hrami s `recommendations > 0`, tedy podíl chybějících labelů.

In [ ]:
year = release.dt.year
has_rec = df["recommendations"] > 0
table = pd.DataFrame({
    "games": zero.groupby(year).size(),
    "zero_share": zero.groupby(year).mean(),
    "missing_share": zero[has_rec].groupby(year[has_rec]).mean(),
}).round(3)
table.loc[2010:]

,games,zero_share,missing_share
release_date,,,
2010,255,0.024,0.016
2011,260,0.000,0.000
2012,322,0.000,0.000
2013,468,0.049,0.050
2014,1524,0.007,0.007
2015,2518,0.007,0.006
2016,4143,0.006,0.001
2017,5927,0.017,0.001
2018,7461,0.037,0.013


#### Poměr recenzí a recommendations po letech
Medián poměru počtu recenzí a `recommendations` u her, které mají obojí. Pokud u novějších let klesá, jsou počty recenzí podhodnocené.

In [ ]:
ok = (n_reviews > 0) & (df["recommendations"] > 0)
ratio = (n_reviews[ok] / df.loc[ok, "recommendations"]).groupby(year[ok]).median()
ratio.loc[2010:]

release_date
2010    1.292327
2011    1.430769
2012    1.427679
2013    1.426334
2014    1.312638
2015    1.258172
2016    1.222421
2017    1.140727
2018    1.125000
2019    1.120000
2020    1.098271
2021    1.085107
2022    1.061644
2023    1.059151
2024    0.989707
2025    0.814953
dtype: float64

## 7. Rozhodnutí pro čištění
Implementováno ve funkci `clean()` v `prepare_data.py`.

1. **Datum vydání do roku 2023** – poměr recenzí a `recommendations` je do roku 2023 stabilní, od roku 2024 klesá (sekce 6).
2. **Hry s aspoň 1 recenzí** – nulu nelze odlišit od chybějících dat (sekce 1 a 6).
3. **Aspoň 3 screenshoty a `header_image`** – obrázky jsou vstupem modelu (sekce 3).
4. **Bez softwaru** – záznamy pouze se softwarovými žánry nejsou hry (sekce 3).
5. **Bez playtestů a duplicit** – u stejné hry pod více `app_id` zůstává záznam s nejvíce recenzemi (sekce 5).
6. **Oprava jazyků** – z chybně zapsaných položek (HTML/BBCode zbytky, více jazyků v jedné položce, poznámky v závorkách) se názvy jazyků vytáhnou regulárním výrazem (sekce 3).
7. **Vstupy pouze známé před vydáním** – recenze, `recommendations`, vlastníci, CCU, herní doba, `tags` a DLC vznikají až po vydání (leakage).

## 8. Vyčištěná data

#### Počet her po jednotlivých krocích

In [ ]:
from prepare_data import clean

df_clean = clean(df)

#### Hry bez žánrů, kategorií nebo jazyků po čištění

In [ ]:
(df_clean[["genres", "categories", "supported_languages"]].map(len) == 0).sum()

#### Jazyky po opravě

In [ ]:
langs_clean = df_clean["supported_languages"].explode().value_counts()
print("Počet unikátních hodnot:", len(langs_clean))
display(langs_clean.tail(20))

#### Hry s téměř všemi jazyky
Nejvzácnější jazyky mají skoro stejný počet her – pokud jde o stejné hry, uvádějí téměř všechny jazyky, které Steam nabízí.

In [ ]:
print("Hry s 90+ jazyky:", (df_clean["supported_languages"].map(len) >= 90).sum())